In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Functions.Utils import *
from Functions.Graphs import *
from Functions.RTLO_R2 import *
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
from sklearn.metrics import root_mean_squared_error as rmse

def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler

def prepare_data(sig, n, m):
    X, Y = [], []
    # O shift (s) é calculado para alinhar o final de Y com a predição futura
    # Seguindo sua lógica: se n=4, m=3 -> Y começa no índice 2 (hi_3)
    s = n - m + 1 
    
    for i in range(len(sig) - n - 1):
        X.append(sig[i : i + n])
        Y.append(sig[i + s : i + s + m])
        
    return np.array(X), np.array(Y)

path = r'Datasets\DEVRT\NISSAN LEAF\20230421_NISSAN_DONOSTIA_ULIA_056.csv'
df = pd.read_csv(path)
df.fillna(0, inplace=True)
df.head()


pwr = (df['Motor Pwr(w)'].values)
spd = (df['speed'].values)
rpm = (df['rpm'].values)
trq = (df['Torque Nm'].values)
df2 = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'power': pwr,})


In [2]:
pwr_dm = NormalizeSeries(df['Motor Pwr(w)'].values)
spd = NormalizeSeries(df['speed'].values)
rpm = NormalizeSeries(df['rpm'].values)
trq = NormalizeSeries(df['Torque Nm'].values)
#PlotTwoScalesPLY(y1=pwr_dm,y2=trq,y1_name='Motor Power (w)',y2_name='Torque (Nm)')
PlotSeriesPLY(ySeries=[pwr_dm,spd,rpm,trq],names=['Motor Power (w)','Speed (km/h)','RPM','Torque (Nm)'])

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

# 1. Configuração do dispositivo (Usa sua RTX 4060 se o CUDA estiver ativo)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando no dispositivo: {device}")


# Para o treino funcionar bem, vamos inflar esses dados repetindo-os (simulando uma série longa)
df2 = pd.concat([df2] * 50, ignore_index=True)
df2['power'] += np.random.normal(0, 50, size=len(df2)) # adiciona um ruído para o treino

# Separando Inputs (X) e Output (y)
X_raw = df2[['speed', 'rpm', 'torque']].values
y_raw = df2['power'].values.reshape(-1, 1)

# --- 3. Pré-processamento e Normalização ---
# LSTMs são muito sensíveis à escala dos dados (especialmente com RPM na casa dos 20k e torque negativo)
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

# Função para criar janelas temporais (Sequências deslizantes)
def criar_sequencias(X, y, time_steps=3):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

# Vamos usar as últimas 3 linhas (passos de tempo) para prever a próxima potência
LOOKBACK = 3 
X_seq, y_seq = criar_sequencias(X_scaled, y_scaled, time_steps=LOOKBACK)

# Conversão para tensores do PyTorch
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
y_tensor = torch.tensor(y_seq, dtype=torch.float32)

# Criação do DataLoader para gerenciar os lotes (Batches) de treino
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

# --- 4. Construção da Arquitetura LSTM ---
class PowerPredictorLSTM(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=64, num_layers=2, output_dim=1):
        super(PowerPredictorLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Camada LSTM
        # batch_first=True indica que a entrada possui o formato: [Batch, Time_Steps, Features]
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        
        # Camada de saída Totalmente Conectada (Linear)
        self.linear = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Inicialização dos estados ocultos (h0) e de célula (c0)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        
        # Saída da LSTM: out terá o formato [Batch, Time_Steps, Hidden_Dim]
        out, _ = self.lstm(x, (h0, c0))
        
        # Pegamos apenas o último passo de tempo da sequência para passar para a camada linear
        out = self.linear(out[:, -1, :])
        return out

# Inicializa o modelo
model = PowerPredictorLSTM(input_dim=3, hidden_dim=64, num_layers=2, output_dim=1).to(device)

# --- 5. Critério de Erro e Otimizador ---
criterion = nn.MSELoss() # Erro Quadrático Médio, ideal para regressão de potência
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# --- 6. Loop de Treinamento ---
model.train()
print("\nIniciando o treinamento da LSTM...")
for epoch in range(40):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        # Envia os lotes de dados para o mesmo dispositivo da rede (GPU/CPU)
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Backward pass e Otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_X.size(0)
        
    total_loss = epoch_loss / len(dataloader.dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Época [{epoch+1}/40] | Erro MSE: {total_loss:.6f}")

# --- 7. Teste de Predição com Inversão de Escala ---
model.eval()
with torch.no_grad():
    # Pegamos a primeira janela de teste do próprio dataset para validar
    exemplo_entrada = X_tensor[0:1].to(device) 
    predicao_normalizada = model(exemplo_entrada).cpu().numpy()
    
    # Desnormaliza o resultado para voltar a unidade real de Potência (Watts ou kW)
    predicao_real = scaler_y.inverse_transform(predicao_normalizada)
    valor_real = scaler_y.inverse_transform(y_tensor[0:1].numpy())
    
    print("-" * 50)
    print(f"Potência Real Esperada: {valor_real[0][0]:.2f}")
    print(f"Potência Predita pela LSTM: {predicao_real[0][0]:.2f}")

Rodando no dispositivo: cuda

Iniciando o treinamento da LSTM...
Época [1/40] | Erro MSE: 0.002218
Época [5/40] | Erro MSE: 0.000172
Época [10/40] | Erro MSE: 0.000128
Época [15/40] | Erro MSE: 0.000111


KeyboardInterrupt: 

# Parameters

In [ ]:
eta_fc = 1
eta_bat = 1
rho_H2 = 120*1e6 # J/kg
mH2_dot = pwr_dm/(eta_fc*eta_bat*rho_H2) # kg/s

PlotSeriesPLY(ySeries=[pwr_dm,mH2_dot])

In [ ]:
import numpy as np

class MarkovChainTimeSeries:
    def __init__(self, n_states=7):
        self.n_states = n_states
        self.bins = None
        self.state_means = None
        self.transition_matrix = None

    def fit(self, data):
        """
        Treina a Cadeia de Markov com divisão de estados robusta para dados nulos/esparsos.
        """
        data = np.array(data, dtype=float)
        
        # Para dados com muitos zeros (como pwr_dm), o linspace baseado no min/max
        # é mais estável que percentis para evitar bins com larguras nulas.
        self.bins = np.linspace(np.min(data), np.max(data), self.n_states + 1)
        
        # Ajuste fino das bordas para evitar erros de truncamento
        self.bins[0] -= 1e-5
        self.bins[-1] += 1e-5

        # Calcula o valor médio real de cada estado para de-quantização
        self.state_means = np.zeros(self.n_states)
        for i in range(self.n_states):
            mask = (data >= self.bins[i]) & (data < self.bins[i+1])
            if np.any(mask):
                self.state_means[i] = np.mean(data[mask])
            else:
                # Se o estado estiver vazio, assume o centro geométrico do bin
                self.state_means[i] = (self.bins[i] + self.bins[i+1]) / 2.0

        # Mapeia a série contínua para os estados discretos
        states = np.digitize(data, self.bins) - 1
        states = np.clip(states, 0, self.n_states - 1)

        # Monta a matriz de transição por contagem
        self.transition_matrix = np.zeros((self.n_states, self.n_states))
        for t in range(len(states) - 1):
            self.transition_matrix[states[t], states[t+1]] += 1

        # Normalização com proteção contra linhas zeradas (divisão por zero)
        row_sums = self.transition_matrix.sum(axis=1, keepdims=True)
        
        # Onde a soma da linha for > 0, divide. Onde for 0, distribui probabilidade uniforme (1/n)
        self.transition_matrix = np.where(
            row_sums > 0, 
            self.transition_matrix / row_sums, 
            1.0 / self.n_states
        )

    def predict_next_steps(self, x_k, steps=3):
        if self.transition_matrix is None:
            raise ValueError("Treine o modelo com .fit() primeiro.")

        # Encontra o estado inicial de x_k
        state_k = np.digitize([x_k], self.bins)[0] - 1
        state_k = np.clip(state_k, 0, self.n_states - 1)

        predictions_values = []
        
        # Vetor de estado como distribuição de probabilidade inicial
        state_vector = np.zeros(self.n_states)
        state_vector[state_k] = 1.0

        for step in range(steps):
            # Avança um passo no tempo multiplicando pela matriz de transição
            state_vector = np.dot(state_vector, self.transition_matrix)
            
            # Valor esperado contínuo para o passo k+n
            expected_value = np.sum(state_vector * self.state_means)
            predictions_values.append(expected_value)

        return predictions_values

In [ ]:
modelo_markov = MarkovChainTimeSeries(n_states=3)
modelo_markov.fit(X)

In [ ]:


# Prediz os próximos 3 passos a partir do valor inicial (1640)
valores_preditos = modelo_markov.predict_next_steps(X[-2], steps=1)

print("Amostras originais iniciais:    ", X[-3:])
print("Valores Preditos (x_1, x_2, x_3):", [round(v, 2) for v in valores_preditos])

Amostras originais iniciais:     [    0     0 20760]
Valores Preditos (x_1, x_2, x_3): [3836.8]


In [ ]:
import numpy as np

class OptimizedMarkovChain:
    def __init__(self, n_states=10):
        self.n_states = n_states
        self.state_centers = None
        self.transition_matrix = None

    def _optimize_states_kmeans(self, data, max_iter=100):
        """
        Otimiza a localização dos estados (centros) minimizando o erro de quantização.
        Abordagem K-means 1D robusta.
        """
        # Inicialização dos centros espalhados uniformemente entre o min e max
        centers = np.linspace(np.min(data), np.max(data), self.n_states)
        
        for _ in range(max_iter):
            # Encontra o centro mais próximo para cada ponto
            distances = np.abs(data[:, np.newaxis] - centers)
            labels = np.argmin(distances, axis=1)
            
            # Atualiza os centros com a média dos pontos atribuídos a eles
            new_centers = np.array([
                np.mean(data[labels == i]) if np.sum(labels == i) > 0 else centers[i]
                for i in range(self.n_states)
            ])
            
            # Critério de parada se os centros convergirem
            if np.allclose(centers, new_centers):
                break
            centers = new_centers
            
        return sorted(centers), labels

    def fit(self, data):
        data = np.array(data, dtype=float)
        
        # 1. Otimiza as fronteiras e centros usando K-means para diminuir o erro
        self.state_centers, states = self._optimize_states_kmeans(data)
        self.state_centers = np.array(self.state_centers)

        # 2. Monta a matriz de transição
        self.transition_matrix = np.zeros((self.n_states, self.n_states))
        for t in range(len(states) - 1):
            self.transition_matrix[states[t], states[t+1]] += 1

        # 3. Normalização estocástica com proteção contra divisões por zero
        row_sums = self.transition_matrix.sum(axis=1, keepdims=True)
        self.transition_matrix = np.where(
            row_sums > 0, 
            self.transition_matrix / row_sums, 
            1.0 / self.n_states
        )

    def predict_next_steps(self, x_k, steps=3):
        # Identifica a qual estado otimizado o valor atual pertence
        state_k = np.argmin(np.abs(self.state_centers - x_k))
        
        predictions_values = []
        state_vector = np.zeros(self.n_states)
        state_vector[state_k] = 1.0

        for _ in range(steps):
            state_vector = np.dot(state_vector, self.transition_matrix)
            # O valor esperado minimiza o erro quadrático médio (MSE) da predição
            expected_value = np.sum(state_vector * self.state_centers)
            predictions_values.append(expected_value)

        return predictions_values

    def evaluate_rmse(self, data):
        """ Calcula o erro RMSE de 1 passo à frente em todo o histórico """
        erros_quadraticos = []
        for i in range(len(data) - 1):
            pred = self.predict_next_steps(data[i], steps=1)[0]
            erros_quadraticos.append((data[i+1] - pred) ** 2)
        return np.sqrt(np.mean(erros_quadraticos))

In [ ]:
X = pwr_dm

for n in [5, 7, 10, 15, 20]:
    modelo = OptimizedMarkovChain(n_states=n)
    modelo.fit(X)
    rmse = modelo.evaluate_rmse(X)
    print(f"Número de Estados: {n:2d} | Erro RMSE: {rmse:.2f}")

Número de Estados:  5 | Erro RMSE: 5736.40
Número de Estados:  7 | Erro RMSE: 5660.57
Número de Estados: 10 | Erro RMSE: 5616.98
Número de Estados: 15 | Erro RMSE: 5614.73
Número de Estados: 20 | Erro RMSE: 5611.79


C:\Users\Claudio\AppData\Local\Temp\ipykernel_26744\1847211417.py:51: RuntimeWarning: invalid value encountered in divide
  self.transition_matrix / row_sums,


In [ ]:
eta_dc = 1

soc_o = 0
soc_k = 0.6
soh_0 = 1
soh_k = 0
soh_l = 0
soh_u = 1
Pfc_o = 0
Pfc_k = 0
Pfc_l = 0
Pfc_u = 60 * 1e3
dPfc_k = 0
dPfc_l = -1 * 1e3
dPfc_u = 1 * 1e3
Pbat_o = 0
Pbat_k = 0
Preq_o = 0
Preq_k = 0
dPreq_k = 0
U_dc = 0 + 1e-20
Q_bat = 0 + 1e-20
dT = 0

A_k = np.array([[1, (dT*eta_dc)/(U_dc*Q_bat)],
                [0, 1]]) 
Bu_k = np.array([[(dT*eta_dc)/(U_dc*Q_bat)],
                [1]])
Bv_k = np.array([[(dT)/(U_dc*Q_bat)],
                [0]])
C = np.eye(2)
D = np.array([0,1]).reshape(-1,1)


In [ ]:
i = 0
Preq_k = pwr_dm[i]

In [ ]:
x_k = np.array([soc_k,Pfc_o]).reshape(-1, 1)
v_k = np.array([Preq_k]).reshape(-1, 1)
v_k

array([[1640]], dtype=int64)